<a id='1-2'></a>
### 
This function is designed to handle multiple input messages in a conversational context. The input format is a dictionary with two keys:

1. 'role' - the role that the message is being passed (usually assistant, system or user)
2. 'content' - the prompt

#### Parameters:

- `messages` (List[Dict]): A list of dictionaries, each containing 'role' and 'content' keys to represent each message in the conversation.
- `max_tokens` (int): Determines token limit for the response.
- `model` (str): Model to be used, default is `"meta-llama/Llama-3.2-3B-Instruct-Turbo"`.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read API keys
openai_api_key = os.getenv("OPENAI_API_KEY")
open_router_api_key = os.getenv("OPENROUTER_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")

# Enable LangChain's tracing to monitor and debug chain executions
os.environ["LANGCHAIN_TRACING_V2"] = "true"

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

if not open_router_api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

if not langsmith_api_key:
    raise ValueError("LANGSMITH_API_KEY not found in .env file")

print("✓ Environment loaded successfully")
print(f"✓ OpenAI API Key: {openai_api_key[:20]}..." if openai_api_key else "✗ OpenAI API Key not found")
print(f"✓ OpenRouter API Key: {open_router_api_key[:20]}..." if open_router_api_key else "✗ OpenRouter API Key not found")
print(f"✓ LangSmith API Key: {langsmith_api_key[:20]}..." if langsmith_api_key else "✗ LangSmith API Key not found")

✓ Environment loaded successfully
✓ OpenAI API Key: sk-proj-Xo-lT2Oxn0cM...
✓ OpenRouter API Key: sk-or-v1-fa41df1a84e...
✓ LangSmith API Key: lsv2_pt_8d4edf0310f0...


In [2]:
from IPython.display import Markdown, display

def print_markdown(text):
    """Display formatted markdown text in Jupyter notebooks"""
    display(Markdown(text))

# Show passed message
print_markdown("""
✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!
""")


✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!


In [4]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

# First, let's create a layout for the houses

def house_info_layout(houses):
    # Create an empty string
    layout = ''
    # Iterate over the houses
    for house in houses:
        # For each house, append the information to the string using f-strings
        # The following way using brackets is a good way to make the code readable as in each line you can start a new f-string that will appended to the previous one
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # Don't forget the new line character at the end!
    return layout

In [5]:
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



In [7]:
def generate_prompt(query, houses):
    # The code made above is modular enough to accept any list of houses, so you could also choose a subset of the dataset.
    # This might be useful in a more complex context where you want to give only some information to the LLM and not the entire data
    houses_layout = house_info_layout(houses)
    # Create a hard-coded prompt. You can use three double quotes (") in this cases, so you don't need to worry too much about using single or double quotes and breaking the code
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}    
             """
    return PROMPT

print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?    
             


In [12]:
from openai import OpenAI

# Initialize the client (uses OPENAI_API_KEY environment variable by default)
client = OpenAI()

def generate_response(prompt: str, role: str):
        # Make a chat completion request
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "your helpful assistant. based given prompt, answer the user query accurately."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=150
    )

    # Print the response
    print(response.choices[0].message.content)

In [13]:
#print_markdown("### Generating response for the query: 'What is the most expensive house?'")
#print_markdown(generate_response(generate_prompt("What is the most expensive house?", houses = house_data), role="user"))
generate_response(generate_prompt("What is the most expensive house?", houses = house_data), role="user")

The most expensive house is located at 456 Elm Avenue, Shelbyville, TN 37160, priced at $320,000.


Retrieval metrics are fundamental in RAG systems, as they provide a way to measure performance. To effectively gauge performance, you need a labeled dataset—one where the answers to specific queries are known—allowing you to compare these results with those generated by your RAG system. In this lab, you will use a pre-labeled dataset and focus on Precision and Recall metrics.


<a id='2'></a>
## 2 - Retrieving metrics

---

Let's explore briefly the most common metrics for retrieval systems: Precision@K and Recall@K.

<a id='2-1'></a>
### 2.1 Precision@K

Precision@K provides an evaluation of the relevancy of the top K retrieved documents. It's calculated as the ratio of relevant documents in the top K results to K (the total number of documents retrieved).

$$\text{Precision@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{K}}$$

where K is the number of documents retrieved.

In [15]:
def precision_at_k(relevant_count, k):
    """
    Calculate the Precision@K for a retrieval system.

    Precision@K is the ratio of relevant documents in the top K retrieved documents
    to the total number K of documents retrieved.

    Args:
        relevant_count (int): Number of relevant documents in the top K results.
        k (int): Total number of documents retrieved (K).

    Returns:
        float: The Precision@K value, or 0.0 if k is zero.
    
    Raises:
        ValueError: If any input is negative.
    """
    if relevant_count < 0 or k < 0:
        raise ValueError("All input values must be non-negative.")
    
    if k == 0:
        return 0.0

    return relevant_count / k

<a id='2-2'></a>
### 2.2 Recall@K

Recall@K evaluates the retrieval system's ability to find all relevant documents from the dataset within the top K results. It's calculated as the ratio of relevant documents in the top K results to the total number of relevant documents in the entire corpus.

$$\text{Recall@K} = \frac{\text{Number of Relevant Documents in Top K}}{\text{Total Number of Relevant Documents in Corpus}}$$

In [1]:
def recall_at_k(relevant_count, total_relevant):
    """
    Calculate the Recall@K for a retrieval system.

    Recall@K is the ratio of relevant documents in the top K retrieved documents
    to the total number of relevant documents in the entire corpus.

    Args:
        relevant_count (int): Number of relevant documents in the top K results.
        total_relevant (int): Total number of relevant documents in the corpus.

    Returns:
        float: The Recall@K value, or 0.0 if total_relevant is zero.
    
    Raises:
        ValueError: If any input is negative.
    """
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError("All input values must be non-negative.")

    if total_relevant == 0:
        return 0.0

    return relevant_count / total_relevant

Chunking plays an important role in information retrieval. For example, when building a vector database from a collection of books, different chunk sizes can serve different purposes. Cataloging entire books as single vectors may help in identifying broad themes, but misses specific details. Chunking closer to the paragraph or sentence level enables the retrieval of specific information or concepts.

Language models typically have limitations on the amount of text they can process at once, known as the "context window." Chunking helps ensure that text inputs remain within these boundaries, allowing models to handle large documents, like novels, by splitting them into smaller sections.

In this ungraded lab you will explore ways of chunking and see how it can impact RAG systems!

In [3]:
from typing import List
import requests
import re

url = "https://raw.githubusercontent.com/progit/progit2/main/book/01-introduction/sections/what-is-git.asc"
source_text = requests.get(url).text

print(source_text[:1000])

[[what_is_git_section]]
=== What is Git?

So, what is Git in a nutshell?
This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.
As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.
Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))

==== Snapshots, Not Differences

The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.
Conceptually, most other systems store information as a list of file-based changes.
These other systems (CVS, Subversion, Perforce, and so o

In [7]:
def get_chunks_fixed_size(text: str, chunk_size: int) -> List[str]:
    """
    Splits a given text into chunks of a specified fixed size.

    Args:
        text (str): The input text to be split into chunks.
        chunk_size (int): The maximum number of words per chunk.

    Returns:
        List[str]: A list of text chunks, each containing up to 'chunk_size' words.
    """
    # Split the input text into individual words
    text_words = text.split()
    
    # Initialize a list to hold the chunks of words
    chunks = []
    
    # Iterate over the word indices in steps of 'chunk_size'
    for i in range(0, len(text_words), chunk_size):
        # Select a sublist of words from 'i' to 'i + chunk_size'
        chunk_words = text_words[i: i + chunk_size]
        
        # Join the selected words into a single string with spaces in between
        chunk = " ".join(chunk_words)
        
        # Add the chunk to the list of chunks
        chunks.append(chunk)
    
    # Return the list of word chunks
    return chunks

def get_chunks_fixed_size_with_overlap(text: str, chunk_size: int, overlap_fraction: float) -> List[str]:
    """
    Splits a given text into chunks of a fixed size with a specified overlap fraction between consecutive chunks.

    Parameters:
    - text (str): The input text to be split into chunks.
    - chunk_size (int): The number of words each chunk should contain.
    - overlap_fraction (float): The fraction of the chunk size that should overlap with the adjacent chunk.
      For example, an overlap_fraction of 0.2 means 20% of the chunk size will be used as overlap.

    Returns:
    - List[str]: A list of chunks (each a string) where each chunk might overlap with its adjacent chunk.
    """

    # Split the text into individual words
    text_words = text.split()
    
    # Calculate the number of words to overlap between consecutive chunks
    overlap_int = int(chunk_size * overlap_fraction)
    
    # Initialize a list to store the resulting chunks
    chunks = []
    
    # Iterate over text in steps of chunk_size to create chunks
    for i in range(0, len(text_words), chunk_size):
        # Determine the start and end indices for the current chunk,
        # taking into account the overlap with the previous chunk
        chunk_words = text_words[max(i - overlap_int, 0): i + chunk_size]
        
        # Join the selected words to form a chunk string
        chunk = " ".join(chunk_words)
        
        # Append the chunk to the list of chunks
        chunks.append(chunk)
    
    # Return the list of chunks
    return chunks


In [8]:
for chosen_size in [5, 25, 100]:
    chunks = get_chunks_fixed_size_with_overlap(source_text, chosen_size, overlap_fraction=0.2)
    # Print outputs to screen
    print(f"\nSize {chosen_size} - {len(chunks)} chunks returned.")
    for i in range(3):
        print(f"Chunk {i+1}: {chunks[i]}")


Size 5 - 281 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git?
Chunk 2: Git? So, what is Git in
Chunk 3: in a nutshell? This is an

Size 25 - 57 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git
Chunk 2: if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to
Chunk 3: you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid

Size 100 - 15 chunks returned.
Chunk 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of

In [9]:
# Split the text into paragraphs
def get_chunks_by_paragraph(source_text: str) -> List[str]:
    return source_text.split("\n\n")

# Split the text by Asciidoc section markers
def get_chunks_by_asciidoc_sections(source_text: str) -> List[str]:
    return source_text.split("\n==")

for marker in ["\n\n", "\n=="]:
    chunks = source_text.split(marker)
    # Print outputs to screen
    print(f"\nUsing the marker: {repr(marker)} - {len(chunks)} chunks returned.")
    for i in range(3):
        print(f"Chunk {i+1}: {repr(chunks[i])}")


Using the marker: '\n\n' - 31 chunks returned.
Chunk 1: '[[what_is_git_section]]\n=== What is Git?'
Chunk 2: "So, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))"
Chunk 3: '==== Snapshots, Not Differences'

Using the marker: '\n==' - 7 chunks returned.
Chunk 1: '[[what_is_git_section]]'
Chunk 2: "= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb,

In [13]:
def mixed_chunking(source_text):
    """
    Splits the given source_text into chunks using a mix of fixed-size and variable-size chunking.
    It first splits the text by Asciidoc markers and then processes the chunks to ensure they are 
    of appropriate size. Smaller chunks are merged with the next chunk, and larger chunks can be 
    further split at the middle or specific markers within the chunk.

    Args:
    - source_text (str): The text to be chunked.

    Returns:
    - list: A list of text chunks.
    """

    # Split the text by Asciidoc marker
    chunks = source_text.split("\n==")

    # Chunking logic
    new_chunks = []
    chunk_buffer = ""
    min_length = 25

    for chunk in chunks:
        new_buffer = chunk_buffer + chunk  # Create new buffer
        new_buffer_words = new_buffer.split(" ")  # Split into words
        if len(new_buffer_words) < min_length:  # Check whether buffer length is too small
            chunk_buffer = new_buffer  # Carry over to the next chunk
        else:
            new_chunks.append(new_buffer)  # Add to chunks
            chunk_buffer = ""

    if len(chunk_buffer) > 0:
        new_chunks.append(chunk_buffer)  # Add last chunk, if necessary

    return new_chunks

In [14]:
mixed_chunks = mixed_chunking(source_text)
for i in range(3):
    print(f"Chunk {i+1}: {repr(mixed_chunks[i])}")

Chunk 1: "[[what_is_git_section]]= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))\n"
Chunk 2: "== Snapshots, Not Differences\n\nThe major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.\nConceptually, most other systems store information as a list of file-based changes.\nThese other systems (CVS, Subv

In [20]:
def get_book_text_objects():
    # Source location
    text_objs = list()
    api_base_url = 'https://api.github.com/repos/progit/progit2/contents/book'  # Book base URL
    chapter_urls = ['/01-introduction/sections', '/02-git-basics/sections']  # List of section URLs

    # Loop through book chapters
    for chapter_url in chapter_urls:
        response = requests.get(api_base_url + chapter_url)  # Get the JSON data for the section files in the chapter

        # Loop through inner files (sections)
        for file_info in response.json():
            if file_info['type'] == 'file':  # Only process files (not directories)
                file_response = requests.get(file_info['download_url'])

                # Build objects including metadata
                chapter_title = file_info['download_url'].split('/')[-3]
                filename = file_info['download_url'].split('/')[-1]
                text_obj = {
                    "body": file_response.text,
                    "chapter_title": chapter_title,
                    "filename": filename
                }
                text_objs.append(text_obj)
    return text_objs

book_text_objs = get_book_text_objects()
print(f"Total text objects retrieved: {len(book_text_objs)}")

Total text objects retrieved: 14


In [21]:
def build_chunk_objs(book_text_obj, chunks):
    """
    Constructs a list of chunk objects from a given book text object 
    and its associated chunks.

    Args:
        book_text_obj (dict): A dictionary containing metadata for the book text, 
                              including 'chapter_title' and 'filename'.
        chunks (list): A list of chunks that represent parts of the book text.

    Returns:
        list: A list of dictionaries, each representing a chunk object 
              with 'chapter_title', 'filename', 'chunk', and 'chunk_index'.
    """
    chunk_objs = list()  # Initialize an empty list to store chunk objects
    
    # Iterate over the chunks with an index
    for i, c in enumerate(chunks):
        # Create a dictionary for each chunk with its associated data
        chunk_obj = {
            "chapter_title": book_text_obj["chapter_title"],  # Chapter title from the book text object
            "filename": book_text_obj["filename"],            # Filename from the book text object
            "chunk": c,                                       # The actual chunk of text
            "chunk_index": i                                  # The index of the chunk in the list
        }
        # Append the constructed chunk object to the list
        chunk_objs.append(chunk_obj)

    # Return the list of chunk objects
    return chunk_objs



In [23]:
# Get multiple sets of chunks - according to chunking strategy
chunk_obj_sets = dict()
for book_text_obj in book_text_objs:
    text = book_text_obj["body"]  # Get the object's text body

    # Loop through chunking strategies:
    for strategy_name, chunks in [
        ["fixed_size_25", get_chunks_fixed_size_with_overlap(text, 25, 0.2)],
        ["fixed_size_100", get_chunks_fixed_size_with_overlap(text, 100, 0.2)],
        ["para_chunks", get_chunks_by_paragraph(text)],
        ["para_chunks_min_25", mixed_chunking(text)]
    ]:
        chunk_objs = build_chunk_objs(book_text_obj, chunks)

        if strategy_name not in chunk_obj_sets.keys():
            chunk_obj_sets[strategy_name] = list()

        chunk_obj_sets[strategy_name] += chunk_objs

In [24]:
print(chunk_obj_sets.keys())

dict_keys(['fixed_size_25', 'fixed_size_100', 'para_chunks', 'para_chunks_min_25'])


In [25]:
chunk_type = 'fixed_size_25' # Change it to check the different chunks!
chunk_obj_sets[chunk_type][0:2]

[{'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': '=== About Version Control (((version control))) What is "`version control`", and why should you care? Version control is a system that records changes to a',
  'chunk_index': 0},
 {'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': 'that records changes to a file or set of files over time so that you can recall specific versions later. For the examples in this book, you will use software',
  'chunk_index': 1}]